<a href="https://colab.research.google.com/github/Zajecia-na-PWr-LR/lista-2-Wojciech-Lingo/blob/main/UczenieMaszynowe_25_26_Lista2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza Zbiorów Danych
Laboratorium polega na analizie eksploracynej oraz wykonaniu redukcji wymiarowości dwóch zbiorów danych. W trakcie ćwiczenia zbadasz wskazane zbiory danych w następujących zadaniach:

1. Dla obu zbiorów danych:
    * Wczytaj zbiór danych. Opisz poszczególne kolumny - jakie zawierają atrybuty, co opisują. Zdecyduj czy któreś z kolumn należy przekształcić.
    * Zweryfikuj, czy w zbiorze występują wartości brakujące i/lub odstające. Zdecyduj jak (i czy) należy je usunąć.
    * Zbadaj korelacje między zmiennymi. Możesz posłużyć się macierzą korelacji.
    * Zwizualizuj najciekawsze/najważniejsze według Ciebie zależności w zbiorze.
2. Tylko dla zbioru Spotify Tracks:
    * Utwórz nową cechę "emocja" na podstawie dostępnych kolumn.
    * Dokonaj redukcji wymiarowości za pomocą metod *filter* oraz *wrapper*.
    * Zwizualizij zbiór za pomocą PCA oraz t-SNE. Sprawdź, jak na wizualizację wpływa normalizacja oraz standaryzacja danych.


## Zaliczenie laboratorium


 Za zadania można uzyskać maksymalnie 10 punktów według poniższej tabeli:

| ID | Zadanie | Zbiór danych | Ilość punktów |
|----|---------|--------------|---------------|
| 1  |Wczytanie zbioru danych. Określenie typów zmiennych. Opis kolumn. | Titanic, Spotify | 1 pkt |
| 2  |Filtracja danych. Usunięcie brakujących wartości. | Titanic, Spotify | 2 pkt|
| 3  |Analiza korelacji między zmiennymi (korzystając m. in. z macierzy korelacji) | Titanic, Spotify | 2 pkt |
| 4  |Przedstawienie wizualizacji (histogramów, pudełkowych) opisujących dane. | Titanic, Spotify | 2 pkt |
| 5  |Inżynieria i redukcja cech. | Spotify |1 pkt|
| 6 | Wizualizacja zbioru przy redukcji wymiarów poprzez PCA / t-SNE. Analiza wyników. | Spotify | 2 pkt. |

Analizę (punkty 1-4) należy przeprowadzić dla obu zbiorów. Redukcja (5-6) powinna zostać wykonana tylko dla zbioru *Spotify Tracks*.

### Pytania pomocnicze:
- Co decyduje o jakości i trudności zbioru danych? Czy któryś ze zbiorów z ćwiczenia jest łatwiejszy/trudniejszy? Dlaczego?
- Jakie informacje daje nam analiza pojedynczych cech w przeciwieństwie do analizy wielowymiarowej?
- Jakie własności zbioru mogą stanowić problem dla analizy?
- Na czym polega detekcja wartości odstających? Jaki wpływ na wyniki ma wybrana metoda?
- Jakie są wady/zalety metod radzenia sobie z brakującymi wartościami?
- Jak działa PCA i kiedy warto go stosować?
- Jak działa t-SNE i kiedy warto go stosować? Jaka jest fundamentalna różnica względem PCA?
- Na czym polega standaryzacja danych oraz normalizacja danych? Jakie są różnice
pomiędzy tymi metodami?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Analiza zbioru danych [*Titanic*](https://www.kaggle.com/competitions/titanic/overview)

10 kwietnia 1912 roku brytyjski transatlantyk Titanic wypływa z Southampton, a 5 dni później schodzi na dno Atlantyku. Z 2208 osób na pokładzie, ocalało jedynie 704 [[1](https://pl.wikipedia.org/wiki/RMS_Titanic#Liczba_ofiar)]. Szanse przeżycia były silnie uzależnione od płci czy klasy podróży.

Zbiór danych Titanic zawiera informacje o 891 pasażerach statku. Podaje on między innymi płeć, klasę podróży, czy numer biletu. Celem tej części listy jest przeanalizowanie zbioru, opisanie wartości w nim występujących, i odpowiedź na pytanie: kto miał największe szanse na przeżycie Titanica?

## Opis danych

In [ ]:
# wczytanie zbioru danych
titanic = pd.read_csv("titanic.csv")

titanic.head(10)

In [ ]:
titanic.info()

(titanic.isnull().mean() * 100).sort_values(ascending=False)

# Opis danych:
# PassengerId – identyfikator pasażera (niepotrzebny)
# Survived – czy pasażer przeżył (0/1, nie/tak) – zmienna docelowa
# Pclass – klasa biletu (1, 2, 3) – zmienna kategoryczna
# Name – imię i nazwisko - do usunięcia
# Sex – płeć – kategoria (do zamiany na liczby, ale na wykresie ładniej widać jak są kategorie)
# Age – wiek – 20% to braki, ale nie do usunięcia
# SibSp – liczba rodzeństwa/małżonków na pokładzie
# Parch – liczba rodziców/dzieci na pokładzie
# Ticket – numer biletu (niepotrzbeny)
# Fare – cena biletu – zmienna numeryczna
# Cabin – numer kabiny – do usunięcia 77% to braki
# Embarked – port zaokrętowania – kategoria kilka braków (do zmiany naliczby ale znowu dla
#   człowieka lepiej widać jak sie nie zmieni)

# Wnioski:
# uzupełnić wartości (Age, Embarked)
# zmienne (Sex, Embarked) zamienić na numeryczne, żeby lepiej działało
# usunąć kolumny (PassengerId, Name, Ticket, Cabin)

## Przekształcenie danych

In [ ]:
titanic['Sex'] = titanic['Sex'].replace({'male': 0, 'female': 1}).astype('Int64')
titanic['Embarked'] = titanic['Embarked'].replace({'C': 0, 'Q': 1, 'S': 2}).astype('Int64')

## Brakujące wartości

Wskazówki:
- Wartości ciągłe możemy zinterpolować korzystając z gotowej metody [`pandas.DataFrame.interpolate`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.interpolate.html)
- Wartości dyskretne można uzupełnić konkretną wartością używając metody [`pandas.DataFrame.fillna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)
- Aby lepiej ocenić czym uzupełnić NaNy, warto wyświetlić kolumnę na wykresie.
- W przypadku dyskretnych wartości, warto również znaleźć wartości unikatowe funkcją `unique()`.

### Wiek pasażera

In [ ]:
titanic['Age'] = titanic['Age'].fillna(
    titanic.groupby(['Pclass'])['Age'].transform('median')
)


In [ ]:
assert titanic['Age'].isnull().values.any() == False, "Kolumna 'wiek' zawiera brakujące wartości"


### Zaokrętowanie

In [ ]:
titanic['Embarked'] = titanic.groupby(['Pclass','Sex'])['Embarked']\
                             .transform(lambda x: x.fillna(x.mode()[0]))

In [ ]:
assert titanic['Embarked'].isnull().values.any() == False, "Kolumna 'zaokrętowanie' zawiera brakujące wartości"

### Kabina

In [ ]:
titanic['Cabin'] = titanic['Cabin'].fillna('Unknown')

In [ ]:
assert titanic['Cabin'].isnull().values.any() == False, "Kolumna 'kabina' zawiera brakujące wartości"

In [ ]:
assert titanic.isnull().values.any() == False, "Zbiór danych zawiera brakujące wartości"

## Przedstawienie danych na wykresach

Wybierz 3-4 wykresy które przekazują według Ciebie najwięcej informacji.

In [ ]:

sns.countplot(data=titanic, x='Sex', hue='Survived')
plt.title('Przeżywalność wg płci')
plt.show()

sns.countplot(data=titanic, x='Pclass', hue='Survived')
plt.title('Przeżywalność wg klasy pasażera')
plt.show()

sns.countplot(data=titanic, x='Embarked', hue='Survived')
plt.title('Przeżywalność wg portu zaokrętowania')
plt.show()

sns.boxplot(data=titanic, x='Survived', y='Age')
plt.title('Rozkład wieku wśród przeżyłych i zmarłych')
plt.show()

plt.figure(figsize=(10,8))
corr = titanic.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm")
plt.title("Macierz korelacji cech utworów")
plt.show()

## Podsumowanie - ocena przeżywalności

Na podstawie informacji uzyskanych podczas ćwiczenia - kto miał największe szanse przeżyć Titanica? Jaka cecha (bądź zestaw cech) decydowały o wyniku podróży?

In [ ]:
#największą szanse miała kobieta w młodym wieku która wsiadła w porcie C i podróżowała w klasie 1

# Największe szanse na przeżycie mieli: kobiety, pasażerowie 1 klasy oraz dzieci.
# Kluczowe cechy decydujące o wyniku podróży to: Sex, Pclass i Age.

# Analiza zbioru danych [*Spotify Tracks*](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)

Celem tej części listy jest analiza dużego, rzeczywistego zbioru danych zawierającego informacje o ponad 100 tysiącach piosenek ze Spotify. Zbiór zawiera kilkanaście cech numerycznych opisujących utwór oraz cechy kategoryczne: wykonawcę, nazwę albumu, gatunek.

Analiza zbioru pozwoli w późniejszym etapie na skuteczną redukcję wymiarowości za pomocą dwóch metod: *filter* oraz *wrapper*. Końcowym celem listy jest przedstawienie wielowymiarowego zbioru na dwuwymiarowym wykresie za pomocą `PCA` oraz `tSNE`.

## Opis danych

In [ ]:
spotify = pd.read_csv("spotify.csv")

spotify.shape
spotify.describe()
spotify.head(20)

In [ ]:
spotify.describe()
spotify.info()

# Każdy wiersz odpowiada jednemu utworowi, a kolumny opisują jego cechy.
# id – unikalny identyfikator utworu w Spotify
# name – nazwa utworu
# artists – wykonawca/wykonawcy utworu (lista zapisana jako tekst)
# popularity – popularność utworu 0–100, im wyższa tym większa popularność
# year – rok wydania utworu
# release_date – dokładna data wydania (YYYY-MM-DD lub tylko rok)
# duration_ms – długość utworu w milisekundach (można zamienić na min)
# explicit – informacja czy utwór zawiera treści niecenzuralne (0 – nie, 1 – tak)
# danceability – taneczność utworu (0.000 – 1.000)
# energy – energia utworu (0.000 – 1.000)
# loudness – głośność utworu w decybelach (wartości ujemne)
# speechiness – współczynnik ilości słów (0.000 – 1.000)
# acousticness – utwór jest akustyczny (0.000 – 1.000)
# instrumentalness – utwór jest instrumentalny (0.000 – 1.000)
# liveness –  żywość utworu (0.000 – 1.000)
# valence – pozytywność utworu (0.000 – 1.000)
# tempo – tempo utworu w BPM
# key – tonacja utworu (0–11)
# mode – tryb (0 – molowy, 1 – durowy)

## Przekształcenia i filtracja danych

In [ ]:
spotify["duration_min"] = spotify["duration_ms"] / 60000
spotify = spotify.drop("duration_ms", axis=1)
spotify["release_date"] = pd.to_datetime(spotify["release_date"], errors="coerce")
spotify = spotify.drop_duplicates(subset=["id"])
spotify = spotify.dropna()
spotify.head()

## Wizualizacje

In [ ]:

spotify["danceability_group"] = pd.cut(spotify["danceability"], bins=10)
plt.figure(figsize=(10,5))
sns.boxplot(data=spotify, x="danceability_group", y="popularity")
plt.title("Popularność utworów w zależności od poziomu danceability")
plt.xlabel("Przedziały danceability")
plt.ylabel("Popularity")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(data=spotify, x="explicit", y="popularity")
plt.title("Popularność utworów a obecność wulgaryzmów")
plt.xlabel("Explicit (0 = brak wulgaryzmów, 1 = wulgaryzmy)")
plt.ylabel("Popularity")
plt.show()

plt.figure(figsize=(10,8))
corr = spotify.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm")
plt.title("Macierz korelacji cech utworów")
plt.show()

## Dodanie nowej cechy - emocja

Emocje w muzyce są przekazywane za pomocą akordów. W zbiorze mamy dostępne informacje nt. klucza i mody piosenki. Ich kombinacja będzie odpowiadać emocji, zgodnie z [tą rozpiską](https://ledgernote.com/blog/interesting/musical-key-characteristics-emotions/).

Moda w zbiorze jest określona jako 0 lub 1, co odpowiada *minor* i odpowiednio *major*.

Klucz jest w [notacji liczbowej](https://en.wikipedia.org/wiki/Pitch_class), czyli 0 odpowiada **C**, 1 odpowiada **C#**, itd.

Twoim zadaniem jest dodanie nowej kolumny "emotion" na podstawie dostępnych informacji.

In [ ]:
# dla ułatwienia - gotowe słowniki

key_map = {0: 'C', 1: 'C#', 2: 'D', 3: 'D#', 4: 'E', 5: 'F', 6: 'F#', 7: 'G', 8: 'G#', 9: 'A', 10: 'A#', 11: 'B'}

emotion_map = {
    ('C', 'Major'):  'Happy',
    ('C#', 'Major'): 'Joyful',
    ('D', 'Major'):  'Triumphant',
    ('D#', 'Major'): 'Cruel',
    ('E', 'Major'):  'Noisy',
    ('F', 'Major'):  'Passionate',
    ('F#', 'Major'): 'Bright',
    ('G', 'Major'):  'Rustic',
    ('G#', 'Major'): 'Rich',
    ('A', 'Major'):  'Pastoral',
    ('A#', 'Major'): 'Magnificent',
    ('B', 'Major'):  'Harsh',

    ('C', 'Minor'):  'Sad',
    ('C#', 'Minor'): 'Melancholic',
    ('D', 'Minor'):  'Pensive',
    ('D#', 'Minor'): 'Anxious',
    ('E', 'Minor'):  'Grieving',
    ('F', 'Minor'):  'Tragic',
    ('F#', 'Minor'): 'Gloomy',
    ('G', 'Minor'):  'Serious',
    ('G#', 'Minor'): 'Mournful',
    ('A', 'Minor'):  'Tender',
    ('A#', 'Minor'): 'Dark',
    ('B', 'Minor'):  'Lonely',
}

In [ ]:
spotify["key_name"] = spotify["key"].map(key_map)
spotify["mode_name"] = spotify["mode"].map({0: "Minor", 1: "Major"})

spotify["emotion"] = list(
    map(lambda x, y: emotion_map.get((x, y)), spotify["key_name"], spotify["mode_name"])
)

spotify[["key", "mode", "key_name", "mode_name", "emotion"]].head(10)

## Redukcja wymiarowości

W tej części zadania należy:
- zredukować wymiary zbioru poprzez usunięcie wybranych kolumn korzystając z metod *filter* i *wrapper*
- zwizualizować zbiór danych korzystając z metod redukcji wymiarowości
- zaimplementować standaryzację oraz normalizację
- przeanalizować jak te działania wpływają na wyniki redukcji



### Filter
Analizując pojedyncze kolumny, zdecyduj czy któreś z nich należy usunąć.

In [ ]:

cols_to_drop = ["id", "name", "artists", "release_date", "mode_name", "key_name","danceability_group","duration_ms"]
spotify_filtered = spotify.drop(columns=cols_to_drop, errors="ignore")


print("Kolumny po zastosowaniu filtra:")
print(spotify_filtered.columns.tolist())

### Wrapper
Korzystając z gotowej implementacji klasyfikatora las losowy, zdecyduj czy któreś z kolumn należy usunąć.

**UWAGA**

To jest bardzo uproszczona implementacja wrappera, która ma na celu jedynie pokazać jego działanie.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

def classify(df, features):
    """
    Dostępne cechy:
        'valence', 'year', 'acousticness', 'danceability',
        'energy', 'explicit', 'instrumentalness', 'key',
        'liveness', 'loudness', 'mode', 'popularity',
        'speechiness', 'tempo', 'duration_min', 'emotion'
    """
    df = df.drop(columns=["Unnamed: 0", "track_id", "artists", "album_name", "track_name"], errors="ignore")
    df["explicit"] = df["explicit"].astype(int)
    df = df.sample(10000, random_state=42).dropna()

    df["popularity_bracket"] = pd.cut(df["popularity"], bins=[0, 33, 66, 100], labels=["low", "mid", "high"])
    df = df.dropna(subset=["popularity_bracket"])
    y = LabelEncoder().fit_transform(df["popularity_bracket"])
    X = df[features].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42))
    ])
    pipe.fit(X_train, y_train)

    acc  = pipe.score(X_test, y_test)

    print(f"Features  : {features}")
    print(f"Test Acc  : {acc:.4f}")
    return acc

In [ ]:
# WYWOŁAJ KLASYFIKATOR W TYM MIEJSCU
example_feats = ["explicit", "danceability","energy","loudness","speechiness", "acousticness","instrumentalness", "liveness", "valence"]

base_acc = classify(spotify, example_feats)


### Wizualizacje (PCA i t-SNE)

W wizualizacji przetestuj kilka kolumn jako docelowe.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn import preprocessing

In [ ]:
# przykładowe funkcje

def dataframe_xy(df):
    raise NotImplementedError()
    return X, y

# normalize to [0,1] range
def normalize(X):
    raise NotImplementedError()

# standarize (e.g, w/ scikit standard scaler)
def standarize(X):
    raise NotImplementedError()

# remove outliers
def remove_outliers(X, y):
    raise NotImplementedError()
